# Thử một batch SFT trên GPU
Chọn runtime **GPU T4** trước khi chạy. Notebook xác minh source và bundle dữ liệu, sau đó đo một lượt forward/backward của mẫu train dài đã biết. Không optimizer step, không lưu checkpoint, không xem test và không gọi đây là M1 đầy đủ.

In [ ]:
from pathlib import Path

import torch
from google.colab import drive

assert torch.cuda.is_available(), "Chọn GPU T4 trong Thời gian chạy > Thay đổi loại thời gian chạy"
print("GPU:", torch.cuda.get_device_name(0))
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/Vietnamese_VQA_NLE")
assert ROOT.is_dir(), f"Sai tài khoản Drive hoặc thiếu thư mục dự án: {ROOT}"
print("Drive:", ROOT)

In [ ]:
import hashlib
import json
import subprocess
import sys
import zipfile

SOURCE_DIR = ROOT / "source/releases"
hash_files = sorted(SOURCE_DIR.glob("source_*.sha256"))
assert hash_files, f"Không có source release: {SOURCE_DIR}"
SOURCE_HASH = hash_files[-1]
SOURCE_NAME = SOURCE_HASH.stem
SOURCE_ZIP = SOURCE_DIR / f"{SOURCE_NAME}.zip"
expected, filename = SOURCE_HASH.read_text(encoding="utf-8").split()
assert filename == SOURCE_ZIP.name
digest = hashlib.sha256()
with SOURCE_ZIP.open("rb") as source:
    for block in iter(lambda: source.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == expected, "Source ZIP checksum mismatch"
WORK_SOURCE = Path("/content/vqa_sources") / SOURCE_NAME
REPO = WORK_SOURCE / "Vietnamese-VQA-Explanation"
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    names = archive.namelist()
    assert names and all(
        name.startswith("Vietnamese-VQA-Explanation/") and ".." not in Path(name).parts
        for name in names
    )
    assert archive.testzip() is None
    manifest = json.loads(archive.read("Vietnamese-VQA-Explanation/SOURCE_MANIFEST.json"))
    if REPO.exists():
        assert all(
            hashlib.sha256((REPO / name).read_bytes()).hexdigest() == value
            for name, value in manifest.items()
        ), "Source đã giải nén không khớp"
    else:
        WORK_SOURCE.mkdir(parents=True, exist_ok=False)
        archive.extractall(WORK_SOURCE)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO),
     "transformers>=4.49,<5", "peft>=0.14,<1", "accelerate>=1.2,<2",
     "bitsandbytes>=0.45,<1"],
    check=True,
)
print("Source verified:", SOURCE_NAME, expected)

In [ ]:
BUNDLE = ROOT / "datasets/releases/vivqax_image_disjoint_v1"
EXPECTED_BUNDLE_SHA256 = "46846a81c7b3594d4fc9670ec51019cfd652f1eb169af7978e39dc5a40a55522"
WORK_DATA = Path("/content/vqa_work/vivqax_image_disjoint_v1")
assert hashlib.sha256((BUNDLE / "bundle.json").read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
bundle = json.loads((BUNDLE / "bundle.json").read_text(encoding="utf-8"))
command = [sys.executable, str(REPO / "scripts/verify_coco_bundle.py"), str(BUNDLE)]
if not WORK_DATA.exists():
    command += ["--extract-to", str(WORK_DATA)]
subprocess.run(command, cwd=REPO, check=True)
assert len(list((WORK_DATA / "images").glob("*.jpg"))) == bundle["total_images"]
checksums = WORK_DATA / "metadata/full_image_checksums.csv"
assert hashlib.sha256(checksums.read_bytes()).hexdigest() == bundle["image_checksums_sha256"]
assert (WORK_DATA / "metadata/all.jsonl").is_file()
print("Data verified:", bundle["total_images"], "images")

In [ ]:
RUN_ROOT = ROOT / "runs"
before = {path.name for path in RUN_ROOT.glob("sft-batch-pilot_*")}
subprocess.run(
    [sys.executable, "-u", str(REPO / "scripts/run_pipeline.py"),
     "sft-batch-pilot", "--environment",
     "configs/environments/colab_full.example.yaml"],
    cwd=REPO, check=True,
)
created = [p for p in RUN_ROOT.glob("sft-batch-pilot_*") if p.name not in before]
assert len(created) == 1, f"Không xác định được run mới: {created}"
run = created[0]
execution = json.loads((run / "execution.json").read_text(encoding="utf-8"))
report = json.loads((run / "batch_pilot.json").read_text(encoding="utf-8"))
assert execution["status"] == "passed" and execution["returncode"] == 0
assert report["status"] == "passed" and report["question_id"] == "335717000"
print("RUN:", run)
for key in ("gpu", "tokens", "loss", "max_lora_gradient",
            "forward_backward_seconds", "peak_allocated_gib", "peak_reserved_gib"):
    print(f"{key}: {report[key]}")